In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

#Machine learnings Modules
from sklearn.linear_model import Lasso
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import Lasso
from sklearn.pipeline import Pipeline

#Models
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor

from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

import warnings
warnings.filterwarnings("ignore")

%matplotlib inline

In [ ]:
df = pd.read_csv("../data/service_type_large.csv")

df.head()

In [ ]:
df = df.rename(columns={
    "Yil": "year",
    "Ay": "month",
    "Hizmet Turu": "service_type",
    "Muracaat Sayisi": "application_count",
    "Yasli Nufus Orani": "elderly_ratio",
    "Saglik Enflasyonu": "health_inflation",
    "Birim Maliyet": "unit_cost",
    "Toplam Harcama": "total_amount"
})

In [ ]:
features = [
    'year',
    'month',
    'service_type',
    'application_count',
    'elderly_ratio',
    'health_inflation',
    'unit_cost'
]

target = 'total_amount'

df = df[features + [target]].dropna()

X = df[features]
y = df[target]

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
#Categorical variables -> one hot encoding
#Numerical variables -> standard scaler, mean= 0, std=1 
categorical_features = ['service_type']
numeric_features = ['year', 'month', 'application_count', 'elderly_ratio', 'health_inflation', 'unit_cost']

preprocessor = ColumnTransformer(transformers=[
    ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features),
    ('num', StandardScaler(), numeric_features)
])

In [ ]:
#Preprocessing and lineer regression
lr_model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', LinearRegression())
])

lr_model.fit(X_train, y_train)
y_pred_lr = lr_model.predict(X_test)

In [ ]:
#Simplified random forest to prevent overfitting
rf_simplified = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', RandomForestRegressor(
        n_estimators=100,
        max_depth=10,            # 5 yerine 10 deneyebilirsin
        min_samples_leaf=5,      # 10 yerine 5
        max_features='sqrt',
        random_state=42
    ))
])

#Train
rf_simplified.fit(X_train, y_train)

#Prediction
y_pred_rf_simple = rf_simplified.predict(X_test)

In [ ]:
#Lasso model is defined within the pipeline
lasso_model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', Lasso(alpha=1.0, random_state=42, max_iter=10000))  # max_iter önemlidir!
])

#Train
lasso_model.fit(X_train, y_train)

#Prediction
y_pred_lasso = lasso_model.predict(X_test)

In [ ]:
#Performance measurements
results = pd.DataFrame({
    "Model": ["Linear Regression", "Random Forest", "Lasso Regression"],
    "R2 Score": [
        r2_score(y_test, y_pred_lr),
        r2_score(y_test, y_pred_rf),
        r2_score(y_test, y_pred_lasso)
    ],
    "RMSE": [
        np.sqrt(mean_squared_error(y_test, y_pred_lr)),
        np.sqrt(mean_squared_error(y_test, y_pred_rf)),
        np.sqrt(mean_squared_error(y_test, y_pred_lasso))
    ],
    "MAE": [
        mean_absolute_error(y_test, y_pred_lr),
        mean_absolute_error(y_test, y_pred_rf),
        mean_absolute_error(y_test, y_pred_lasso)
    ]
})

print(results)

In [ ]:
#Linear Regression graphic
plt.figure(figsize=(8, 5))
plt.scatter(y_test, y_pred_lr, alpha=0.6)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], '--r')
plt.xlabel("Gerçek Değerler")
plt.ylabel("Tahmin Edilen Değerler")
plt.title("Linear Regression – Gerçek vs Tahmin")
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
#Random Forest graphic
plt.figure(figsize=(8, 5))
plt.scatter(y_test, y_pred_rf, alpha=0.6)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], '--r')
plt.xlabel("Gerçek Değerler")
plt.ylabel("Tahmin Edilen Değerler")
plt.title("Random Forest – Gerçek vs Tahmin")
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
#Lasso Regression graphic
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(6, 6))
sns.scatterplot(x=y_test, y=y_pred_lasso, alpha=0.6)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], color='red', linestyle='--')
plt.xlabel("Gerçek Değerler")
plt.ylabel("Tahmin Edilen Değerler")
plt.title("Lasso Regression – Gerçek vs Tahmin")
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
#List of all feature names
ohe = preprocessor.named_transformers_['cat']
encoded_feature_names = list(ohe.get_feature_names_out(['service_type']))

all_features = numeric_features + encoded_feature_names

In [ ]:
#Feature coefficients table
lr_inner_model = lr_model.named_steps['regressor']
coefficients = lr_inner_model.coef_

coef_df = pd.DataFrame({
    'Feature': all_features,
    'Coefficient': coefficients
}).sort_values(by='Coefficient', ascending=False)

In [ ]:
#Linear Regression – Feature Coefficients graphic
plt.figure(figsize=(10, 6))
sns.barplot(x='Coefficient', y='Feature', data=coef_df, palette='coolwarm')
plt.axvline(0, color='gray', linestyle='--')
plt.title('Linear Regression – Feature Coefficients')
plt.xlabel('Katsayı Değeri')
plt.ylabel('Değişkenler')
plt.tight_layout()
plt.show()

In [ ]:
#Retrieving categorical feature names from the preprocessing pipeline
ohe = preprocessor.named_transformers_['cat']
encoded_feature_names = list(ohe.get_feature_names_out(['service_type']))

#All feature names
all_features = numeric_features + encoded_feature_names

In [ ]:
#Random Forest pipeline model
rf_inner_model = rf_model.named_steps['regressor']

#Importance values
importances = rf_inner_model.feature_importances_

#DataFrame 
importance_df = pd.DataFrame({
    'Feature': all_features,
    'Importance': importances
}).sort_values(by='Importance', ascending=False)

In [ ]:
#Random Forest – Feature Importance graphic
plt.figure(figsize=(10, 6))
sns.barplot(x='Importance', y='Feature', data=importance_df, palette="viridis")
plt.title('Random Forest – Feature Importance')
plt.xlabel('Önemi')
plt.ylabel('Değişkenler')
plt.tight_layout()
plt.show()

In [ ]:
#Get coefficients from model in pipeline
lasso_coef = lasso_model.named_steps['regressor'].coef_

#Get feature names in order
feature_names = preprocessor.get_feature_names_out()

#DataFrame
coef_df = pd.DataFrame({
    'Feature': feature_names,
    'Coefficient': lasso_coef
})

#Sort to separate the non-zero and the major contributors
coef_df = coef_df.sort_values(by='Coefficient', key=lambda x: abs(x), ascending=False)

#Lasso Regression – Feature Coefficients graphic
plt.figure(figsize=(10, 6))
sns.barplot(x='Coefficient', y='Feature', data=coef_df)
plt.title("Lasso Regression – Feature Coefficients")
plt.tight_layout()
plt.show()

In [ ]:
#Zero coefficients in the Lasso model
zero_coef = (lasso_coef == 0).sum()
total_coef = len(lasso_coef)
print(f"Lasso modeli toplam {total_coef} özelliğin {zero_coef} tanesini sıfırladı.")

In [ ]:
#Random Forest cross validation
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_val_score

#Pipeline: Preprocessing + Random Forest
rf_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', RandomForestRegressor(random_state=42))
])

#Cross-validation
scores = cross_val_score(rf_pipeline, X, y, cv=5, scoring='neg_mean_squared_error')

print(f"Mean CV MSE: {-scores.mean():.2f}")
print(f"Std CV MSE: {scores.std():.2f}")

In [ ]:
#Random Forest learning curve - for overfit test
from sklearn.model_selection import learning_curve
import numpy as np
import matplotlib.pyplot as plt

train_sizes, train_scores, test_scores = learning_curve(
    estimator=rf_pipeline,
    X=X,
    y=y,
    train_sizes=np.linspace(0.1, 1.0, 10),
    cv=5,
    scoring='neg_mean_squared_error',
    n_jobs=-1
)

train_mean = -np.mean(train_scores, axis=1)
test_mean = -np.mean(test_scores, axis=1)

plt.figure(figsize=(8, 5))
plt.plot(train_sizes, train_mean, 'o-', label="Training error")
plt.plot(train_sizes, test_mean, 'o-', label="Validation error")
plt.xlabel("Training set size")
plt.ylabel("MSE")
plt.title("Learning Curve - Random Forest with Pipeline")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
#Train Pipeline
rf_pipeline.fit(X, y)

#Permutation importance 
result = permutation_importance(rf_pipeline, X, y, n_repeats=10, random_state=42, n_jobs=-1)

#OneHot + Scaler - real feature names
#Get Preprocessor
preprocessor_fitted = rf_pipeline.named_steps['preprocessor']

#Categorical Variables After OneHot 
cat_encoder = preprocessor_fitted.named_transformers_['cat']
cat_feature_names = cat_encoder.get_feature_names_out(categorical_features)

#Numerical Variables After Scaler
all_feature_names = np.concatenate([cat_feature_names, numeric_features])

#Length control
print("len(feature_names):", len(all_feature_names))
print("len(importances):", len(result.importances_mean))

In [ ]:
preprocessor_fitted = rf_pipeline.named_steps['preprocessor']

#Get transformed X
X_transformed = preprocessor_fitted.transform(X)

#Feature count
print("X_transformed shape:", X_transformed.shape)

#Match feature names
true_feature_names = np.concatenate([
    cat_encoder.get_feature_names_out(categorical_features),
    numeric_features
])[:len(result.importances_mean)]  # 🔥 En kritik satır

importance_df = pd.DataFrame({
    'Feature': true_feature_names,
    'Importance Mean': result.importances_mean,
    'Importance Std': result.importances_std
}).sort_values(by='Importance Mean', ascending=False)

display(importance_df)

In [ ]:
#Simplified random forest scores
from sklearn.model_selection import cross_val_score
import numpy as np

cv_scores = cross_val_score(
    rf_simplified, 
    X, y,
    scoring='neg_mean_squared_error',
    cv=5,
    n_jobs=-1
)

#Turn the positive
cv_mse_scores = -cv_scores
print(f"Mean CV MSE: {cv_mse_scores.mean():,.2f}")
print(f"Std CV MSE: {cv_mse_scores.std():,.2f}")